# Data Description
- Background: 使用者要求補抓議員與Meta資料；保存CEC登記PDF、MOI現任名錄、立院本屆名錄及Meta存取回應。
- date received: 20261018
- Path to data file: 官方線上來源詳見 `data/raw/online_20261018/capture_manifest.json` 與 `meta_access_manifest.json`。
- Unit of observation: PDF的頁／表／列；HTML的名錄項目；manifest的一次存取。
- Sample period: 2026登記名冊（PDF製表2026-09-07）、2026-10-18擷取的官方名錄與存取狀態；非完整競選窗口。
- Known issues: 原始HTML可能只有頁框、驗證挑戰或官方說明；HTTP成功不等於取得貼文。MOI任期未列；立院現任、離職不可混算。PDF表格JSON是來源轉換，非官方原始JSON。
- Definition for each variable:
    - CEC原欄：選舉區、登記日期（民國）、姓名、推薦之政黨、備註。
    - MOI名錄：姓名、縣市、機關／職稱、黨籍、詳細資訊連結；是擷取時來源列示值。
    - 立院：姓名、個人頁、黨徽網址；依所在標題區分現任與離職，不從黨徽像素推定黨籍。
    - PDF轉表JSON：page為1起算頁碼；tables是逐頁原始儲存格巢狀陣列，未刪除換行。
    - capture manifest：path本機相對路徑、url來源、captured_date擷取日期、sha256檔案指紋、status下載狀態。
    - Meta manifest：subject對象、url網址、http_status回應碼、return_code程式退出碼、error錯誤、path回應檔、sha256指紋、interpretation解讀、posts_collected未知、complete_window=False。
- Data path / SHA-256: 下方 INPUT_HASHES 逐檔固定；每列代表一個實際檔案。
- Role: raw-source / source-conversion / access-response（依檔案標示）。
- Verification status: 本Notebook乾淨kernel執行，逐檔存在／hash／格式檢查；原始名錄不等於研究樣本。
- Next processing notebook: [資料整理](../code/prepare_council_and_meta_20261018.ipynb)；PDF來源轉換程式 `data/code/extract_council_registration.py`。


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


## Change for Your Own Data
以下為本次確定的精確來源路徑與版本；來源轉表JSON亦明列，原始PDF不變。

In [2]:
INPUT_HASHES = {'data/raw/online_20261018/capture_manifest.json': '3add11cf37c58de398b6a9c54e384d80fb8c23911d28cce021a3886652f6fa3f', 'data/raw/online_20261018/cec_council_county.pdf': '11d6334e4a5cf3bc7539455ef047ac40f62e250d3c151348683ab1141f83d517', 'data/raw/online_20261018/cec_council_county_tables.json': '04e15c58f7e6af08dfada94ee643bb94fe4610833e5c471f41c050b7e36fce79', 'data/raw/online_20261018/cec_council_metropolitan.pdf': '28a9bf7edeb4b2f45a0b3ca473d59c930fcf8cba575c8879fb33397815116090', 'data/raw/online_20261018/cec_council_metropolitan_tables.json': '645a70d9312d41be408b1ba0579f12924f000e0b10b9d44f5f5b4047eb04bea2', 'data/raw/online_20261018/ly_members.html': '54202822c014d09a5e78fa68309906661345ae3e553fb1210d6b75192c77951a', 'data/raw/online_20261018/meta_access_01.html': 'da0d3c1a3eaf9fbd40707aeef61e4eee804017beff4006e9c3f048dd371dcc6d', 'data/raw/online_20261018/meta_access_02.html': '19e33623156a1cf3d10ba4ce835413c80c0a201a98b311c2eeac49a4a1cbceca', 'data/raw/online_20261018/meta_access_03.html': 'fc0b32adb923ca383f12d936331df5a6e259df1fa990b56c2f9946ab8af436ea', 'data/raw/online_20261018/meta_access_04.html': '6d6f3b5e5971aa42770ad0910ba16a9fd30fcca825c65e356777be8b3790fc6e', 'data/raw/online_20261018/meta_access_05.html': 'e6654daaf8515a41a17c6d96444dd436f366b35f7cf502b150fe9bdee30cde26', 'data/raw/online_20261018/meta_access_06.html': '500e367cd7c691907071116246e71842f8d9628c8f506dafb0f1c4876b95dda7', 'data/raw/online_20261018/meta_access_07.html': '166db55b42133547f09b02bb17f1e4832dbcb1e8555496cd4223a6d3a1871077', 'data/raw/online_20261018/meta_access_08.html': '4f607fed0095311033066f512f02b9d16fd5de4b660810a586fdefc556b259be', 'data/raw/online_20261018/meta_access_09.html': 'bbf55ba001f1f6d36df884aa37287dcd207db9a1dced3f4351eb04706673bc46', 'data/raw/online_20261018/meta_access_10.html': '86ec64ff936e0df7f72000abf0d05b08de7febac8471086ab5fa4e984b1af2d1', 'data/raw/online_20261018/meta_access_11.html': '0a7c8f615e6da11adcba6fbe04726a6193cf72d06db1f5834ae78c39919cada4', 'data/raw/online_20261018/meta_access_12.html': '9bf0f39436349994f5d3e2043154352f2440fe9b757df1cff4a66de8c6383691', 'data/raw/online_20261018/meta_access_13.html': '7cf17bc7d6755f450de95c24d4b061a85b058b96c4fcc2ac94517cbb383af939', 'data/raw/online_20261018/meta_access_manifest.json': 'b8069512d839d2708487303f8c6b08cdf43d2776032288299c5fdf09b736b93a', 'data/raw/online_20261018/moi_county.html': '7ba34c8d17af8cf5bad108312e3218e5f3e5444fef457966f19ec199b97b50b5', 'data/raw/online_20261018/moi_metropolitan.html': '41294443945db5603a7e664434b7532ff58fbc75f1ab63aa6d7258d06486c5af'}
for rel, expected in INPUT_HASHES.items():
    p=project_file(rel,'raw')
    assert sha256_file(p)==expected, rel
print('Hash checks passed:',len(INPUT_HASHES))


Hash checks passed: 22


## Summary


In [3]:
import json
from bs4 import BeautifulSoup
records=[]
for rel,expected in INPUT_HASHES.items():
    p=PROJECT_ROOT/rel
    record={'path':rel,'bytes':p.stat().st_size,'sha256':expected,'role':'source_snapshot'}
    if p.suffix=='.pdf':
        assert p.read_bytes().startswith(b'%PDF-')
        record['format_check']='PDF signature'
    elif p.suffix=='.json':
        payload=json.loads(p.read_text());record['items']=len(payload)
        record['format_check']='JSON parsed'
        if rel.endswith('_tables.json'):record['role']='source_conversion_not_official_json'
    else:
        soup=BeautifulSoup(p.read_bytes(),'html.parser')
        record['title']=soup.title.get_text(' ',strip=True) if soup.title else ''
        record['format_check']='HTML parsed; body may be shell'
        if 'meta_access_' in p.name:record['role']='access_response_not_post_data'
    records.append(record)
df=pd.DataFrame(records).fillna('')
display(df)


,path,bytes,sha256,role,items,format_check,title
0,data/raw/online_20261018/capture_manifest.json,1496,3add11cf37c58de398b6a9c54e384d80fb8c23911d28cc...,source_snapshot,5.0,JSON parsed,
1,data/raw/online_20261018/cec_council_county.pdf,644908,11d6334e4a5cf3bc7539455ef047ac40f62e250d3c1513...,source_snapshot,,PDF signature,
2,data/raw/online_20261018/cec_council_county_ta...,133739,04e15c58f7e6af08dfada94ee643bb94fe4610833e5c47...,source_conversion_not_official_json,44.0,JSON parsed,
3,data/raw/online_20261018/cec_council_metropoli...,409781,28a9bf7edeb4b2f45a0b3ca473d59c930fcf8cba575c88...,source_snapshot,,PDF signature,
4,data/raw/online_20261018/cec_council_metropoli...,91317,645a70d9312d41be408b1ba0579f12924f000e0b10b9d4...,source_conversion_not_official_json,22.0,JSON parsed,
5,data/raw/online_20261018/ly_members.html,357763,54202822c014d09a5e78fa68309906661345ae3e553fb1...,source_snapshot,,HTML parsed; body may be shell,立法院全球資訊網 -本屆立委
6,data/raw/online_20261018/meta_access_01.html,227949,da0d3c1a3eaf9fbd40707aeef61e4eee804017beff4006...,access_response_not_post_data,,HTML parsed; body may be shell,Meta Content Library and API | Transparency Ce...
7,data/raw/online_20261018/meta_access_02.html,503109,19e33623156a1cf3d10ba4ce835413c80c0a201a98b311...,access_response_not_post_data,,HTML parsed; body may be shell,New Tools to Support Independent Research
8,data/raw/online_20261018/meta_access_03.html,5495,fc0b32adb923ca383f12d936331df5a6e259df1fa990b5...,access_response_not_post_data,,HTML parsed; body may be shell,Just a moment...
9,data/raw/online_20261018/meta_access_04.html,513548,6d6f3b5e5971aa42770ad0910ba16a9fd30fcca825c65e...,access_response_not_post_data,,HTML parsed; body may be shell,廣告檔案庫 API


### Sample


In [4]:
display(df.head(10))
pages=json.loads((PROJECT_ROOT/'data/raw/online_20261018/cec_council_metropolitan_tables.json').read_text())
display(pd.DataFrame(pages[0]['tables'][0][1:11],columns=pages[0]['tables'][0][0]))


,path,bytes,sha256,role,items,format_check,title
0,data/raw/online_20261018/capture_manifest.json,1496,3add11cf37c58de398b6a9c54e384d80fb8c23911d28cc...,source_snapshot,5.0,JSON parsed,
1,data/raw/online_20261018/cec_council_county.pdf,644908,11d6334e4a5cf3bc7539455ef047ac40f62e250d3c1513...,source_snapshot,,PDF signature,
2,data/raw/online_20261018/cec_council_county_ta...,133739,04e15c58f7e6af08dfada94ee643bb94fe4610833e5c47...,source_conversion_not_official_json,44.0,JSON parsed,
3,data/raw/online_20261018/cec_council_metropoli...,409781,28a9bf7edeb4b2f45a0b3ca473d59c930fcf8cba575c88...,source_snapshot,,PDF signature,
4,data/raw/online_20261018/cec_council_metropoli...,91317,645a70d9312d41be408b1ba0579f12924f000e0b10b9d4...,source_conversion_not_official_json,22.0,JSON parsed,
5,data/raw/online_20261018/ly_members.html,357763,54202822c014d09a5e78fa68309906661345ae3e553fb1...,source_snapshot,,HTML parsed; body may be shell,立法院全球資訊網 -本屆立委
6,data/raw/online_20261018/meta_access_01.html,227949,da0d3c1a3eaf9fbd40707aeef61e4eee804017beff4006...,access_response_not_post_data,,HTML parsed; body may be shell,Meta Content Library and API | Transparency Ce...
7,data/raw/online_20261018/meta_access_02.html,503109,19e33623156a1cf3d10ba4ce835413c80c0a201a98b311...,access_response_not_post_data,,HTML parsed; body may be shell,New Tools to Support Independent Research
8,data/raw/online_20261018/meta_access_03.html,5495,fc0b32adb923ca383f12d936331df5a6e259df1fa990b5...,access_response_not_post_data,,HTML parsed; body may be shell,Just a moment...
9,data/raw/online_20261018/meta_access_04.html,513548,6d6f3b5e5971aa42770ad0910ba16a9fd30fcca825c65e...,access_response_not_post_data,,HTML parsed; body may be shell,廣告檔案庫 API


,選舉區,登記日期,姓名,推薦之政黨,備註
0,臺北市第1選舉區,115/08/31,謝閔弘,時代力量,
1,臺北市第1選舉區,115/08/31,李文,無,
2,臺北市第1選舉區,115/09/02,陳彥廷,無,
3,臺北市第1選舉區,115/09/02,鍾佩玲,民主進步黨,
4,臺北市第1選舉區,115/09/02,紀建漢,無,
5,臺北市第1選舉區,115/09/02,陳賢蔚,民主進步黨,
6,臺北市第1選舉區,115/09/02,林世宗,民主進步黨,
7,臺北市第1選舉區,115/09/02,黃瀞瑩,台灣民眾黨,
8,臺北市第1選舉區,115/09/02,林揚庭,中國國民黨,
9,臺北市第1選舉區,115/09/02,陳慈慧,民主進步黨,


### Summary Stats


In [5]:
display(df.groupby('role').size().rename('files').to_frame()); display(df['bytes'].describe().to_frame())
print('Bytes describe saved responses, not candidate or post counts.')

,files
role,
access_response_not_post_data,13
source_conversion_not_official_json,2
source_snapshot,7


,bytes
count,22.000000
mean,394976.272727
std,220523.627804
min,1496.000000
25%,260402.500000
50%,488801.000000
75%,490321.500000
max,805729.000000


Bytes describe saved responses, not candidate or post counts.
